In [ ]:
search_sample_size = 20_000
rf_n_iter = 8
models_to_run = ["logreg", "random_forest"]
run_temporal = True
save_to_mongo = True
reuse_results = False
cache_name = "results.joblib"
rf_trees_temporal = 100
run_leakage = True

In [ ]:
import sys; sys.path.append("..")
import config
config.SEARCH_SAMPLE_SIZE = search_sample_size
config.RF_N_ITER = rf_n_iter

import joblib
import pandas as pd
from src.db_sql import load_clean_data
from src.features import get_xy, split_data
from src.models import tune_and_fit, fit_baseline
from src.evaluation import evaluate_model
from src.temporal import temporal_split, fit_temporal
from src.db_mongo import save_params, save_metrics, save_predictions, save_leakage
from src.logger_setup import flush_mongo_logs

from src.leakage_check import leakage_check

df = load_clean_data()
X, y = get_xy(df)
X_train, X_test, y_train, y_test = split_data(X, y)

if run_leakage:
    leak = leakage_check(df)
    print(leak[["variante", "accuracy", "f1_macro", "delta_vs_modelo_real"]].round(3))
    if save_to_mongo:
        save_leakage(leak)
flush_mongo_logs()


2026-11-10 02:01:40,363 | INFO     | src.db_sql | Tabla 'datos_limpios' leída: 174116 filas, 20 columnas
2026-11-10 02:01:41,764 | INFO     | src.features | X: (174116, 12) | y: (174116,) | predictoras: ['M2', 'AMBIENTES', 'ANTIGUEDAD', 'BAÑOS', 'BAULERA', 'COCHERA', 'LAVADERO', 'TERRAZA', 'LAT', 'LON', 'BARRIO', 'ORIENT']
2026-11-10 02:01:43,813 | INFO     | src.features | Train: 139292 filas | Test: 34824 filas
2026-11-10 02:01:44,449 | INFO     | src.features | Target en train:
SEGMENTO
alto     0.333
medio    0.333
bajo     0.333
2026-11-10 02:01:44,458 | INFO     | src.features | Target en test:
SEGMENTO
alto     0.333
medio    0.333
bajo     0.333


In [ ]:
# entrenamiento con caché, para no repetir los 20 minutos del Random Forest:
config.MODELS_DIR.mkdir(exist_ok=True)
RES_PATH = config.MODELS_DIR / cache_name
if reuse_results and RES_PATH.exists():
    results = joblib.load(RES_PATH)
    results = {k: v for k, v in results.items() if k in models_to_run + ["dummy"]}
else:
    results = {n: tune_and_fit(n, X_train, y_train) for n in models_to_run}
    results["dummy"] = fit_baseline(X_train, y_train)
    joblib.dump(results, RES_PATH)

2026-11-10 02:02:43,724 | INFO     | src.models | [logreg] búsqueda de hiperparámetros sobre 20000 filas (GridSearchCV, cv=3, scoring=f1_macro)
2026-11-10 02:04:36,974 | INFO     | src.models | [logreg] mejores parámetros: {'clf__C': 1} | f1_macro en CV: 0.6106 | 113 s
2026-11-10 02:05:00,197 | INFO     | src.models | [logreg] reentrenado con 139292 filas en 23 s
2026-11-10 02:05:00,774 | INFO     | src.models | [random_forest] búsqueda de hiperparámetros sobre 20000 filas (RandomizedSearchCV, cv=3, scoring=f1_macro)
2026-11-10 02:08:46,614 | INFO     | src.models | [random_forest] mejores parámetros: {'clf__n_estimators': 200, 'clf__min_samples_leaf': 10, 'clf__max_features': 0.5, 'clf__max_depth': None} | f1_macro en CV: 0.6365 | 226 s


In [ ]:
# escenario aleatorio

all_metrics, all_preds = [], []
for name, r in results.items():
    m, p = evaluate_model(name, r["model"], X_train, y_train, X_test, y_test)
    all_metrics.append(m); all_preds.append(p)

resumen = pd.DataFrame(all_metrics)[["model", "accuracy", "precision_macro", "recall_macro",
                                     "f1_macro", "accuracy_train_sample"]].round(4)
print(resumen)

In [ ]:
# escenario temporal

t_metrics, t_preds = [], []
if run_temporal:
    Xtr_t, Xte_t, ytr_t, yte_t = temporal_split(df, X, y)
    for name, r in results.items():
        mdl = fit_temporal(name, r["model"], Xtr_t, ytr_t, rf_trees=rf_trees_temporal)
        m, p = evaluate_model(name, mdl, Xtr_t, ytr_t, Xte_t, yte_t)
        t_metrics.append(m); t_preds.append(p)

    print(pd.DataFrame(t_metrics)[["model", "accuracy", "precision_macro",
                                   "recall_macro", "f1_macro"]].round(4))
    pt = pd.concat(t_preds)
    pt["AÑO"] = df.loc[pt["id_fila"], "AÑO"].to_numpy()
    print(pt.groupby(["modelo", "AÑO"])["acierto"].mean().unstack().round(3))

In [ ]:
# persistencia

if save_to_mongo:
    save_params(results)
    save_metrics(all_metrics, "aleatorio")
    save_predictions(pd.concat(all_preds, ignore_index=True), "aleatorio")
    if run_temporal:
        save_metrics(t_metrics, "temporal")
        save_predictions(pd.concat(t_preds, ignore_index=True), "temporal")
flush_mongo_logs()

In [7]:
# prueba leakage

all_metrics, all_preds = [], []
for name, r in results.items():
    m, p = evaluate_model(name, r["model"], X_train, y_train, X_test, y_test)
    all_metrics.append(m); all_preds.append(p)

resumen = pd.DataFrame(all_metrics)[["model", "accuracy", "precision_macro", "recall_macro",
                                     "f1_macro", "accuracy_train_sample"]].round(4)
print(resumen)

print(leakage_check(df))
flush_mongo_logs()

2026-11-10 01:22:04,317 | INFO     | src.evaluation | [logreg] TEST accuracy=0.6189 precision=0.6131 recall=0.6189 f1_macro=0.6151 (train acc 0.6222)
2026-11-10 01:22:04,319 | INFO     | src.evaluation | [logreg] matriz de confusión (filas=real, columnas=predicho ['bajo', 'medio', 'alto']):
[[8191 2560  856]
 [3316 5216 3075]
 [ 898 2565 8147]]
2026-11-10 01:22:07,835 | WARNING  | src.evaluation | [random_forest] posible sobreajuste: accuracy train 0.765 vs test 0.676
2026-11-10 01:22:08,046 | INFO     | src.evaluation | [random_forest] TEST accuracy=0.6763 precision=0.6758 recall=0.6763 f1_macro=0.6760 (train acc 0.7646)
2026-11-10 01:22:08,048 | INFO     | src.evaluation | [random_forest] matriz de confusión (filas=real, columnas=predicho ['bajo', 'medio', 'alto']):
[[8587 2477  543]
 [2642 6448 2517]
 [ 518 2575 8517]]


           model  accuracy  precision_macro  recall_macro  f1_macro  \
0         logreg    0.6189           0.6131        0.6189    0.6151   
1  random_forest    0.6763           0.6758        0.6763    0.6760   

   accuracy_train_sample  
0                 0.6222  
1                 0.7646  


2026-11-10 01:22:21,644 | INFO     | src.leakage_check | Leakage check [sin DOLARES (modelo real)]: accuracy=0.647 f1_macro=0.644
2026-11-10 01:22:36,489 | INFO     | src.leakage_check | Leakage check [sin DOLARES, con AÑO]: accuracy=0.648 f1_macro=0.644
2026-11-10 01:22:50,445 | INFO     | src.leakage_check | Leakage check [con DOLARES]: accuracy=0.689 f1_macro=0.685
2026-11-10 01:22:54,358 | INFO     | src.leakage_check | Leakage check [con DOLARES + AÑO]: accuracy=0.709 f1_macro=0.705


                    variante  accuracy  f1_macro
0  sin DOLARES (modelo real)     0.647     0.644
1       sin DOLARES, con AÑO     0.648     0.644
2                con DOLARES     0.689     0.685
3          con DOLARES + AÑO     0.708     0.705


In [ ]:
# Borrado
# from src.db_mongo import get_db
# db = get_db()
# for c in ("predicciones", "metricas", "parametros"):
#     db[c].drop()